## Institutional Analysis

How do enrollment trends differ between public and private institutions, and is this difference statistically significant?


In [117]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from scipy import stats

In [118]:
enroll_df = enrollment_df = pd.read_csv(os.path.dirname(os.getcwd()) + '/data/cleaned_enrollmentdata.csv')
enrollment_df.rename(columns={'State or jurisdiction': 'State'}, inplace=True)
enrollment_df.head()

,State,Year,Total_Pub_Under,4y_Pub_Under,2y_Pub_Under,Total_Pub_Postbacc,Total_Priv_Under,Np_4y_Priv_Under,Fp_4y_Priv_Under,Np_2y_Priv_Under,Fp_2y_Priv_Under,Total_Priv_Postbacc,Np_4y_Priv_Postbacc,Fp_4y_Priv_Postbacc
0,Alabama,2012-13,216535,130260,86275,34510,49382,21674,24045,518,3145,9884,3917,5967
1,Alabama,2013-14,213669,129045,84624,34615,47519,20808,23295,472,2944,9909,3866,6043
2,Alabama,2014-15,212458,129916,82542,34531,47172,21160,22714,518,2780,10867,4354,6513
3,Alabama,2015-16,212968,131503,81465,34483,44682,20938,21000,436,2308,10826,4593,6233
4,Alabama,2016-17,216115,135080,81035,34923,41893,19891,19627,386,1989,11121,4679,6442


In [119]:
states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut",
    "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
    "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan",
    "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire",
    "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio",
    "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia",
    "Wisconsin", "Wyoming", "District of Columbia"
]

state_enrollment = enrollment_df[enrollment_df["State"].isin(states)]
state_enrollment['State'].unique()

array(['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California',
       'Colorado', 'Connecticut', 'Delaware', 'District of Columbia',
       'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana',
       'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland',
       'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi',
       'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire',
       'New Jersey', 'New Mexico', 'New York', 'North Carolina',
       'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania',
       'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee',
       'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington',
       'West Virginia', 'Wisconsin', 'Wyoming'], dtype=object)

In [120]:
state_enrollment = state_enrollment.copy()
state_enrollment.head()

,State,Year,Total_Pub_Under,4y_Pub_Under,2y_Pub_Under,Total_Pub_Postbacc,Total_Priv_Under,Np_4y_Priv_Under,Fp_4y_Priv_Under,Np_2y_Priv_Under,Fp_2y_Priv_Under,Total_Priv_Postbacc,Np_4y_Priv_Postbacc,Fp_4y_Priv_Postbacc
0,Alabama,2012-13,216535,130260,86275,34510,49382,21674,24045,518,3145,9884,3917,5967
1,Alabama,2013-14,213669,129045,84624,34615,47519,20808,23295,472,2944,9909,3866,6043
2,Alabama,2014-15,212458,129916,82542,34531,47172,21160,22714,518,2780,10867,4354,6513
3,Alabama,2015-16,212968,131503,81465,34483,44682,20938,21000,436,2308,10826,4593,6233
4,Alabama,2016-17,216115,135080,81035,34923,41893,19891,19627,386,1989,11121,4679,6442


**sub-question:** Is the public-to-private enrollment ratio changing over time?

In [121]:
ratio_df = state_enrollment.groupby(['Year']).agg({
    'Total_Pub_Under': 'sum',
    'Total_Priv_Under': 'sum'
}).reset_index()

ratio_df['enroll_ratio'] = ratio_df['Total_Pub_Under'] / ratio_df['Total_Priv_Under']

In [122]:
ratio_df = ratio_df.groupby(['Year']).agg({'enroll_ratio': 'mean'}).reset_index()

baseline_year = ratio_df[state_enrollment['Year'] == '2012-13']
baseline_ratio = ratio_df.loc[
    ratio_df['Year'] == '2012-13',
    'enroll_ratio'
].iloc[0]

total_change_pct = ((ratio_df['enroll_ratio'].iloc[-1] - baseline_ratio) / baseline_ratio) * 100
print(total_change_pct, "%")

7.734349918534489 %


/var/folders/6g/_wpr8bs53rs84lqb6nn_rk900000gn/T/ipykernel_62194/3981090137.py:3: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [123]:
ratio_df['Year_Numeric'] = range(len(ratio_df))

# Now run regression on the aggregated data (ratio_df), not the full dataset
slope, intercept, r_value, p_value, std_err = stats.linregress(
    ratio_df['Year_Numeric'], 
    ratio_df['enroll_ratio']
)


In [124]:
print(f"Slope: {slope:.4f}")
print(f"R-squared: {r_value**2:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Statistically significant trend")
else:
    print("Not statistically significant")

Slope: 0.0411
R-squared: 0.5081
P-value: 0.0207
Statistically significant trend


The change in the public-to-private enrollment ratio is not statistically significant because the p-value is greater than 0.05.

**sub-question:** Are public institutions declining faster/slower than private institutions?


In [125]:
public_df = state_enrollment.groupby(['Year']).agg({
    'Total_Pub_Under': 'sum'
}).reset_index()
public_baseline= public_df[public_df['Year'] == '2012-13']['Total_Pub_Under'].iloc[0]

private_df = state_enrollment.groupby(['Year']).agg({
    'Total_Priv_Under': 'sum'
}).reset_index()
private_baseline= private_df[private_df['Year'] == '2012-13']['Total_Priv_Under'].iloc[0]

public_df['Year_Numeric'] = range(len(public_df))
private_df['Year_Numeric'] = range(len(private_df))

public_slope, public_intercept, public_r_value, public_p_value, public_std_err = stats.linregress(
    public_df['Year_Numeric'], 
    public_df['Total_Pub_Under']
)

private_slope, private_intercept, private_r_value, private_p_value, private_std_err = stats.linregress(
    private_df['Year_Numeric'], 
    private_df['Total_Priv_Under']
)

In [126]:
public_df.head()

,Year,Total_Pub_Under,Year_Numeric
0,2012-13,13458541,0
1,2013-14,13332032,1
2,2014-15,13230125,2
3,2015-16,13130934,3
4,2016-17,13126042,4


In [127]:
print(f"Public slope: {public_slope:.4f}")
print(f"Private slope: {private_slope:.4f}")
print(f"Public R-squared: {public_r_value**2:.4f}")
print(f"Private R-squared: {private_r_value**2:.4f}")
print(f"Public P-value: {public_p_value}")
print(f"Private P-value: {private_p_value}")

Public slope: -136361.8970
Private slope: -87605.7576
Public R-squared: 0.7560
Private R-squared: 0.9253
Public P-value: 0.0010815892084622362
Private P-value: 8.76569440575374e-06


In [128]:
if public_p_value < 0.05:
    print("Public trend is statistically significant")
else:
    print("Public trend is not statistically significant")

if private_p_value < 0.05:
    print("Private trend is statistically significant")
else:
    print("Private trend is not statistically significant")

Public trend is statistically significant
Private trend is statistically significant


Because the p-value for both public and private trends is less than 0.05, both trends are statistically significant. That being said the negative slope for public institutions is a greater negative(steeper slope) implying a faster decline. However the percent changes calculated below are not supportive of that claim.

In [129]:
public_annual_pct = (public_slope / public_baseline) * 100
private_annual_pct = (private_slope / private_baseline) * 100

print(f"\nAnnual percentage change (relative to baseline):")
print(f"  Public: {public_annual_pct:.3f}% per year")
print(f"  Private: {private_annual_pct:.3f}% per year")


Annual percentage change (relative to baseline):
  Public: -1.013% per year
  Private: -2.057% per year


This is the percentage change in enrollment based on a public and private baseline of the first year of data.

In [130]:
public_df['pct_chage'] = public_df['Total_Pub_Under'].pct_change()
private_df['pct_chage'] = private_df['Total_Priv_Under'].pct_change()

public_annual_pct = (public_df['pct_chage'].mean() * 100)
private_annual_pct = (private_df['pct_chage'].mean() * 100)

print(f"\nAnnual percentage change (relative to previous year):")
print(f"  Public: {public_annual_pct:.3f}% per year")
print(f"  Private: {private_annual_pct:.3f}% per year")


Annual percentage change (relative to previous year):
  Public: -1.318% per year
  Private: -2.135% per year


A more accurate statistic of the average percent changer per year. This clearly shows that the private institutions are declining faster than public institutions.

In [131]:
line = pd.DataFrame({
    'Year': public_df['Year'],
    'Public': public_df['Total_Pub_Under'],
    'Private': private_df['Total_Priv_Under']
})

baseline_year = '2012-13'

public_base = line.loc[line['Year'] == baseline_year, 'Public'].iloc[0]
private_base = line.loc[line['Year'] == baseline_year, 'Private'].iloc[0]

line['Public_Index'] = (line['Public'] / public_base) * 100
line['Private_Index'] = (line['Private'] / private_base) * 100

fig = px.line(
    line,
    x='Year',
    y=['Public_Index', 'Private_Index'],
    labels={'value': 'Enrollment Index (Base = 100)'},
    title='Undergraduate Enrollment Change Relative to 2012–13'
)
fig.show()

**sub-question:** What is the effect size of this difference?


In [132]:
en_year = state_enrollment.groupby(['Year']).agg({
    'Total_Priv_Under' : 'sum',
    'Total_Pub_Under' : 'sum'
}).reset_index()
priv_pct_change = en_year['Total_Priv_Under'].pct_change().dropna()
pub_pct_change = en_year['Total_Pub_Under'].pct_change().dropna()

priv_mean = priv_pct_change.mean()
priv_std = priv_pct_change.std()

pub_mean = pub_pct_change.mean()
pub_std = pub_pct_change.std()

# 3️⃣ Compute Cohen's d for percent change (effect size relative to variation)
priv_cohen_d = np.abs(priv_mean / priv_std)
pub_cohen_d = np.abs(pub_mean / pub_std)

# 4️⃣ Print results
print(f"Private enrollment Cohen's d (percent change): {priv_cohen_d:.3f}")
print(f"Public enrollment Cohen's d (percent change): {pub_cohen_d:.3f}")

Private enrollment Cohen's d (percent change): 1.474
Public enrollment Cohen's d (percent change): 0.767


This cohens d shows that the percent change of the private institutions is greater than the public institutions.

**sub-question:** Are there states where private institutions are growing while public decline (or vice versa)?

In [133]:
gr_state = state_enrollment.groupby(['State', 'Year']).agg({
    'Total_Pub_Under': 'sum',
    'Total_Priv_Under': 'sum'
}).reset_index()

In [135]:
gr_state['pub_pct_change'] = gr_state.groupby('State')['Total_Pub_Under'].pct_change().dropna()
gr_state['pub_pct_change'] = gr_state['pub_pct_change'].fillna(0)
gr_state['priv_pct_change'] = gr_state.groupby('State')['Total_Priv_Under'].pct_change().dropna()
gr_state['priv_pct_change'] = gr_state['priv_pct_change'].fillna(0)

In [138]:
state_pub_avg = gr_state.groupby('State')['pub_pct_change'].mean()
state_priv_avg = gr_state.groupby('State')['priv_pct_change'].mean()

In [139]:
t_stat, p_value = stats.ttest_rel(state_pub_avg, state_priv_avg)
print(f"Paired t-test: t = {t_stat:.3f}, p = {p_value:.3f}")

Paired t-test: t = -0.648, p = 0.520


In [141]:
mean_diff = state_pub_avg - state_priv_avg
cohen_d = mean_diff.mean() / mean_diff.std()
print(f"Cohen's d between public and private trends: {cohen_d:.3f}")

Cohen's d between public and private trends: -0.091
